# 08 · Human-in-the-Loop: Tool Approvals

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Some tools should never run on a model's say-so alone: sending emails, issuing
refunds, deleting data. The SDK lets you mark a tool as **needing approval**. When
the model calls it, the run **pauses**: it returns immediately with the pending
request, you save the run's state, and hours later (in a different process, if
you like) a human approves or rejects and the run **resumes** where it stopped.

**You will learn**
- `needs_approval`: marking tools that require a human decision
- `result.interruptions`: what a paused run looks like
- `RunState`: approving or rejecting, and resuming with `Runner.run(agent, state)`
- **Saving a paused run to disk** and resuming it from **another process**
- Conditional approval (e.g. only refunds over €100), rejection messages, `always_approve`
- How this compares with a graph framework's general-purpose pause

**Prerequisites:** notebooks 02 and 04.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import json, subprocess, sys, textwrap, pathlib
from agents import Agent, Runner, function_tool, RunState, RunContextWrapper
from common import get_model

model = get_model()
WORK = pathlib.Path('checkpoints')              # scratch folder for this notebook (git-ignored)
WORK.mkdir(exist_ok=True)

## 1 · Why: pause, don't block

The naive approval is a callback that asks a human and waits (`input("OK?")`).
That **blocks**: the program sits in memory until someone answers, and real
approvals take hours. If the process restarts in the meantime, the work is lost.

The SDK does it differently:

```
 Runner.run(agent, "email the customer")
     └─ model calls send_email ── needs approval ──► the run STOPS and returns:
                                                     result.interruptions = [pending send_email call]
 state = result.to_state()          ──► save state.to_json() anywhere (file, database)
                   … minutes, hours, days …
 state = await RunState.from_json(agent, saved)
 state.approve(item)  /  state.reject(item)
 Runner.run(agent, state)           ──► the tool runs (or not), and the agent continues
```

Nothing is blocked while waiting. The paused run is just data.

## 2 · Marking a tool as needing approval

In [ ]:
outbox = []                                     # what REALLY got sent

@function_tool(needs_approval=True)
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a customer."""
    outbox.append(to)
    return f'Email sent to {to}.'


assistant = Agent(name='Assistant', model=model, tools=[send_email],
                  instructions='You help the support team. Send emails when asked.')

result = await Runner.run(assistant, 'Email sam@example.com that their robot ships on Friday.')

print('final_output :', repr(result.final_output))
print('interruptions:', len(result.interruptions))
pending = result.interruptions[0]
print('pending tool :', pending.tool_name, json.loads(pending.raw_item.arguments))
print('outbox       :', outbox, ' ← nothing sent yet')

The run **returned** without a final output. Its `interruptions` list holds the
pending tool call, with the exact arguments the model chose. That's what you
show the human. Nothing has been sent.

## 3 · Approve or reject, then resume

Convert the result into a **`RunState`**, record the decision, and pass the
state back to `Runner.run` in place of the input:

In [ ]:
state = result.to_state()
state.approve(state.get_interruptions()[0])
resumed = await Runner.run(assistant, state)
print('approved →', resumed.final_output)
print('outbox   :', outbox)

In [ ]:
result = await Runner.run(assistant, 'Email all-customers@example.com that we are shutting down the service.')
state = result.to_state()
item = state.get_interruptions()[0]
print('model wants:', item.tool_name, json.loads(item.raw_item.arguments)['to'])

state.reject(item, rejection_message='Not approved: announcements to all customers need legal sign-off.')
resumed = await Runner.run(assistant, state)
print('rejected →', resumed.final_output)
print('outbox   :', outbox, ' ← still only the approved email')

On rejection the tool doesn't run, and the model receives your rejection
message as the tool's result, so it can explain to the user or try something
else.

## 4 · Pausing across processes

The real value is that a paused run is **data**. `state.to_json()` gives a
plain JSON document, which you can store in a file or database, and any process
that has the same agent definitions can load it with `RunState.from_json` and
continue.

Let's prove it with **real separate processes**. The notebook writes a small
app, `refund_app.py`. We run it once to start a refund request (it pauses and
saves `pending.json`), inspect the file, then run it **again as a new process**
to approve and finish:

In [ ]:
(WORK / 'refund_app.py').write_text(textwrap.dedent(f"""
    import asyncio, json, sys
    sys.path.insert(0, {str(SECTION)!r})
    from agents import Agent, Runner, RunState, function_tool
    from common import get_model

    @function_tool(needs_approval=True, description_override='Refund an order.')
    def issue_refund(order_id: str, amount_eur: float) -> str:
        with open('refunds.log', 'a') as f:
            f.write(f'{{order_id}},{{amount_eur}}\\n')
        return f'Refunded {{amount_eur}} EUR for order {{order_id}}.'

    agent = Agent(name='Billing', model=get_model(), tools=[issue_refund],
                  instructions='You are billing support. Use issue_refund for refunds, then confirm to the customer.')

    async def main(command):
        if command == 'start':
            result = await Runner.run(agent, 'Order B-52 arrived broken. Please refund the 180 EUR.')
            if result.interruptions:
                with open('pending.json', 'w') as f:
                    json.dump(result.to_state().to_json(), f)
                call = result.interruptions[0]
                print('PAUSED, waiting for approval of', call.tool_name, call.raw_item.arguments)
        else:
            with open('pending.json') as f:
                state = await RunState.from_json(agent, json.load(f))
            for item in state.get_interruptions():
                state.approve(item) if command == 'approve' else state.reject(item)
            result = await Runner.run(agent, state)
            print('FINISHED:', result.final_output)

    asyncio.run(main(sys.argv[1]))
"""))

def app(command):
    out = subprocess.run([sys.executable, 'refund_app.py', command], cwd=WORK, capture_output=True, text=True)
    print(f'$ python refund_app.py {command}   (pid of a brand-new process)\n  {out.stdout.strip() or out.stderr[-400:]}')

for f in ('pending.json', 'refunds.log'):
    (WORK / f).unlink(missing_ok=True)
app('start')

In [ ]:
saved = json.loads((WORK / 'pending.json').read_text())
print('pending.json: a', type(saved).__name__, 'with', len(json.dumps(saved)), 'characters; top-level keys:')
print(' ', list(saved)[:12], '…')
print('refunds.log exists?', (WORK / 'refunds.log').exists(), ' ← nothing refunded yet')

In [ ]:
app('approve')                                  # a NEW process loads the state and resumes
print('refunds.log:', (WORK / 'refunds.log').read_text().strip())

The first process exited while the run was paused. The second process knew
nothing except the JSON file and the agent definitions, and it finished the run.
That's how approvals work in a real app: a web request starts the run and stores
the state; a reviewer's click (a different request, maybe days later) loads it,
records the decision and resumes.

**What must match:** the resuming process needs agents and tools with the
**same names**, because the saved state refers to them by name. The saved state
also contains the conversation, so treat the file as sensitive data.

## 5 · Conditional approval

Approving every call is tedious, and people soon approve without reading.
`needs_approval` can be a **function** deciding per call. Here, refunds up to
€100 go through automatically; bigger ones need a human:

In [ ]:
refunds = []

async def over_100(ctx: RunContextWrapper, params: dict, call_id: str) -> bool:
    return params.get('amount_eur', 0) > 100

@function_tool(needs_approval=over_100)
def refund(order_id: str, amount_eur: int) -> str:
    """Refund an order. amount_eur is a whole number of euros."""
    refunds.append((order_id, amount_eur))
    return f'Refunded {amount_eur} EUR for order {order_id}.'

billing = Agent(name='Billing', model=model, tools=[refund],
                instructions='Use the refund tool for refund requests, then confirm briefly.')

for request in ['Refund 40 EUR for order C-1.', 'Refund 900 EUR for order C-2.']:
    r = await Runner.run(billing, request)
    status = f'PAUSED for approval ({len(r.interruptions)} pending)' if r.interruptions else f'done: {r.final_output[:60]}'
    print(f'{request:<32} → {status}')
print('refunds executed:', refunds)

Small refunds went straight through; the big one paused.

**A safety detail we discovered the hard way.** Our first version declared
`amount_eur: float`, and *both* refunds paused. The policy function was
never even called. The model sent `"amount_eur": 40` (an integer), and turning
that into `40.0` would **change** the arguments. The SDK only trusts your policy
when it evaluates exactly the arguments that will be executed; if validation
would alter them, it **fails safe and asks a human**. Two takeaways:

- Declare types that match what the model sends (here `int`), or you'll get
  "approval required" for every call.
- The fail-safe is a feature: the SDK won't let a policy approve arguments that
  differ from the ones that will actually run.

## 6 · Several pending calls, and `always_approve`

If the model requests several approval-gated calls in one turn, they all appear
in `interruptions`, and you decide each one. `state.approve(item,
always_approve=True)` approves this call **and all future calls to that tool
in this run**. It's useful for batch work once a human has checked the first
one. (`reject` has `always_reject=True` too.)

In [ ]:
outbox.clear()
batch = await Runner.run(assistant, 'Email ada@example.com and bo@example.com that the demo moved to 3pm.')
state = batch.to_state()
for item in state.get_interruptions():
    to = json.loads(item.raw_item.arguments)['to']
    decision = 'approve' if to.startswith('ada') else 'reject'
    print(f'  {item.tool_name} → {to}: {decision}')
    if decision == 'approve':
        state.approve(item)
    else:
        state.reject(item, rejection_message='Bo is on leave; do not email.')
final = await Runner.run(assistant, state)
print('agent:', final.final_output[:150])
print('outbox:', outbox)

## 7 · How this compares with a graph framework

| | Agents SDK approvals | LangGraph `interrupt()` |
|---|---|---|
| Where can a run pause? | when an approval-gated **tool** is called | at **any** point in any node |
| What's saved | the run state (conversation + pending calls), when *you* call `to_json()` | a checkpoint after every step, automatically |
| Resume | `Runner.run(agent, state)` | `graph.invoke(Command(resume=...), thread)` |
| Arbitrary questions to a human ("which option?") | model it as a tool that needs approval, or handle it in your own code | built in |

For most agent apps, "approve risky tool calls" is exactly the pause you need,
and the SDK makes it simple. For a pause *between* steps of a code-orchestrated
workflow (notebook 05), just stop your code, store your own progress, and
continue later. That's plain Python, which is the SDK's philosophy.

## 8 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Checking `final_output` without checking `interruptions` | "the agent returned nothing" | always check `result.interruptions` first |
| Keeping paused runs only in memory | restarts lose pending approvals | `state.to_json()` → durable storage |
| Renaming agents or tools while runs are paused | `from_json` can't match the state | keep names stable, or version your agents |
| Approving everything | reviewers stop reading | conditional `needs_approval` for real risk only |
| Showing reviewers a summary instead of the real call | they approve something different | show the tool name and exact arguments |
| Rejecting without a message | the agent can't explain what happened | pass `rejection_message` |

## 9 · Check yourself

<details><summary><b>1.</b> What does <code>Runner.run</code> return when the model calls an approval-gated tool?</summary>

A result with no final output and the pending call(s) in `result.interruptions`. The tool hasn't run.
</details>

<details><summary><b>2.</b> List the steps to approve a pending call and finish the run.</summary>

`state = result.to_state()`, then `state.approve(state.get_interruptions()[0])`, then `await Runner.run(agent, state)`.
</details>

<details><summary><b>3.</b> How can the approval happen in a different process, days later?</summary>

Save `state.to_json()` to durable storage. Later, any process with the same agent definitions calls `RunState.from_json(agent, saved)`, approves or rejects, and runs `Runner.run(agent, state)`.
</details>

<details><summary><b>4.</b> How would you require approval only for refunds above €100?</summary>

Pass a function as `needs_approval`: it receives the context, the call's parameters and the call id, and returns `True` when `amount_eur > 100`.
</details>

<details><summary><b>5.</b> Your workflow must pause *between* two steps for a manager's decision, not at a tool call. What do you do with the SDK?</summary>

Either model the decision as an approval-gated tool, or, in a code-orchestrated workflow, stop after step one, persist your own progress, and run step two when the decision arrives. The SDK doesn't provide a general-purpose pause point the way a graph framework does.
</details>

## Takeaway

Mark risky tools with `needs_approval` (a flag, or a function for conditional
approval). When the model calls one, the run returns with `interruptions`. Turn
it into a `RunState`, save it as JSON if the decision will take a while,
approve or reject, and resume with `Runner.run(agent, state)`, from any process.

Next → `09_sessions_and_state.ipynb`: conversation memory, and exactly what
survives a crash.